<a href="https://colab.research.google.com/github/LarsVoermans/master-thesis-pead/blob/main/XGB%2CRF%2CLR_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
#All imports
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

In [2]:
from google.colab import files
uploaded = files.upload()  # er verschijnt een knop om bestanden te kiezen

Saving train_feature_engineered.parquet to train_feature_engineered.parquet
Saving val_feature_engineered.parquet to val_feature_engineered.parquet


In [3]:
#Loading the dataset
train = pd.read_parquet("train_feature_engineered.parquet")
val = pd.read_parquet("val_feature_engineered.parquet")

In [29]:
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

In [4]:
#making 3 class return
# Train
conditions_train = [
    (train['Return'] > 3),
    (train['Return'] >= -3) & (train['Return'] <= 3),
    (train['Return'] < -3)
]

choices = [0, 1, 2]

train['Return_class_3'] = np.select(conditions_train, choices, default=np.nan)


# Validation
conditions_val = [
    (val['Return'] > 3),
    (val['Return'] >= -3) & (val['Return'] <= 3),
    (val['Return'] < -3)
]

val['Return_class_3'] = np.select(conditions_val, choices, default=np.nan)


In [5]:
#making a return class
target = "Return"

X_train = train.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_train = train["Return_class_3"]

X_val = val.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_val = val["Return_class_3"]


In [6]:
#Dropping colums
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns])
X_val = X_val.drop(columns=[c for c in drop_cols if c in X_val.columns])

In [7]:
#XGB Classifier
# Train+ val
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all.reset_index(drop=True, inplace=True)
y_all.reset_index(drop=True, inplace=True)

# Stratified K-Fold
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
fold_reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):
    print(f"\n--- Fold {fold} ---")

    X_tr, X_val_fold = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_val_fold = y_all.iloc[train_idx], y_all.iloc[val_idx]

    # XGB Classifier
    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        random_state=42
    )

    # Fit
    xgb_model.fit(X_tr, y_tr)

    # Predict
    y_pred = xgb_model.predict(X_val_fold)

    # Scores
    acc = accuracy_score(y_val_fold, y_pred)
    f1 = f1_score(y_val_fold, y_pred, average='macro')
    report = classification_report(y_val_fold, y_pred, output_dict=True)

    accuracy_list.append(acc)
    macro_f1_list.append(f1)
    fold_reports.append(report)

    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")
    print("Classification Report:")
    print(classification_report(y_val_fold, y_pred))

# Average scores
print("\n=== Average over all folds ===")
print(f"Average Accuracy: {np.mean(accuracy_list):.4f}")
print(f"Average Macro F1: {np.mean(macro_f1_list):.4f}")

# Average classification report per class
avg_report = {}
classes = y_all.unique()
for c in classes:
    avg_report[c] = {}
    metrics = ['precision', 'recall', 'f1-score', 'support']
    for m in metrics:
        avg_report[c][m] = np.mean([fold_reports[i][str(c)][m] for i in range(n_splits)])
avg_report['macro avg'] = {m: np.mean([fold_reports[i]['macro avg'][m] for i in range(n_splits)]) for m in metrics}
avg_report['weighted avg'] = {m: np.mean([fold_reports[i]['weighted avg'][m] for i in range(n_splits)]) for m in metrics}

print("\n=== Average Classification Report per class ===")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.6962
Macro F1: 0.4922
Classification Report:
              precision    recall  f1-score   support

         0.0       0.51      0.31      0.39       411
         1.0       0.74      0.93      0.82      1474
         2.0       0.49      0.18      0.27       370

    accuracy                           0.70      2255
   macro avg       0.58      0.48      0.49      2255
weighted avg       0.65      0.70      0.65      2255


--- Fold 2 ---
Accuracy: 0.6943
Macro F1: 0.4912
Classification Report:
              precision    recall  f1-score   support

         0.0       0.53      0.31      0.39       412
         1.0       0.74      0.93      0.82      1473
         2.0       0.45      0.19      0.26       369

    accuracy                           0.69      2254
   macro avg       0.57      0.47      0.49      2254
weighted avg       0.65      0.69      0.65      2254


--- Fold 3 ---
Accuracy: 0.6952
Macro F1: 0.4961
Classification Report:
              preci

In [32]:
y_pred_xgb_oof = []
y_true_xgb_oof = []

for train_idx, val_idx in skf.split(X_all, y_all):

    X_tr = X_all.iloc[train_idx]
    X_val_fold = X_all.iloc[val_idx]

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    model = XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        enable_categorical=True,
        random_state=42
    )

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_val_fold)

    y_pred_xgb_oof.extend(y_pred)
    y_true_xgb_oof.extend(y_val_fold.values)

In [8]:
# -----------------------------
# Random forest
# -----------------------------
X_all = pd.concat([X_train, X_val], axis=0)
y_all = pd.concat([y_train, y_val], axis=0)

X_all = X_all.copy()

cat_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

# -----------------------------
# Stratified K-Fold
# -----------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accuracy_list = []
macro_f1_list = []
reports = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all), 1):

    print(f"\n--- Fold {fold} ---")

    X_tr = X_all.iloc[train_idx].copy()
    X_val_fold = X_all.iloc[val_idx].copy()

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    # -----------------------------
    # Encode categorical features
    # -----------------------------
    if cat_cols:
        encoder = OrdinalEncoder()

        X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
        X_val_fold[cat_cols] = encoder.transform(X_val_fold[cat_cols])

    # -----------------------------
    # Random Forest model
    # -----------------------------
    rf_model = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    )

    # Train
    rf_model.fit(X_tr, y_tr)

    # Predict
    y_pred = rf_model.predict(X_val_fold)

    # Metrics
    acc = accuracy_score(y_val_fold, y_pred)
    macro_f1 = f1_score(y_val_fold, y_pred, average="macro")

    print("Accuracy:", acc)
    print("Macro F1:", macro_f1)
    print(classification_report(y_val_fold, y_pred))

    accuracy_list.append(acc)
    macro_f1_list.append(macro_f1)

    reports.append(classification_report(y_val_fold, y_pred, output_dict=True))

# -----------------------------
# AVerage Results
# -----------------------------
print("\n===== AVERAGE RESULTS =====")
print("Average Accuracy:", np.mean(accuracy_list))
print("Average Macro F1:", np.mean(macro_f1_list))

# -----------------------------
# Average classification report
# -----------------------------
avg_report = {}

for label in reports[0].keys():

    if label == "accuracy":
        avg_report[label] = np.mean([r[label] for r in reports])
        continue

    avg_report[label] = {}

    for metric in reports[0][label].keys():
        avg_report[label][metric] = np.mean([r[label][metric] for r in reports])

print("\nAverage Classification Report per class:")
print(pd.DataFrame(avg_report).T)


--- Fold 1 ---
Accuracy: 0.5436807095343681
Macro F1: 0.48888467550371534
              precision    recall  f1-score   support

         0.0       0.32      0.64      0.43       411
         1.0       0.85      0.54      0.66      1474
         2.0       0.33      0.44      0.38       370

    accuracy                           0.54      2255
   macro avg       0.50      0.54      0.49      2255
weighted avg       0.67      0.54      0.57      2255


--- Fold 2 ---
Accuracy: 0.5532386867790594
Macro F1: 0.4980160843489063
              precision    recall  f1-score   support

         0.0       0.34      0.65      0.45       412
         1.0       0.85      0.55      0.67      1473
         2.0       0.33      0.45      0.38       369

    accuracy                           0.55      2254
   macro avg       0.51      0.55      0.50      2254
weighted avg       0.67      0.55      0.58      2254


--- Fold 3 ---
Accuracy: 0.5346051464063887
Macro F1: 0.47629045731817893
              

In [31]:
cat_cols = X_all.select_dtypes(include=['object', 'category']).columns.tolist()

y_pred_rf_oof = []
y_true_rf_oof = []

for train_idx, val_idx in skf.split(X_all, y_all):

    X_tr = X_all.iloc[train_idx].copy()
    X_val_fold = X_all.iloc[val_idx].copy()

    y_tr = y_all.iloc[train_idx]
    y_val_fold = y_all.iloc[val_idx]

    if cat_cols:
        encoder = OrdinalEncoder()
        X_tr[cat_cols] = encoder.fit_transform(X_tr[cat_cols])
        X_val_fold[cat_cols] = encoder.transform(X_val_fold[cat_cols])

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=6,
        random_state=42,
        class_weight="balanced"
    )

    rf.fit(X_tr, y_tr)
    y_pred = rf.predict(X_val_fold)

    y_pred_rf_oof.extend(y_pred)
    y_true_rf_oof.extend(y_val_fold.values)

In [10]:
#Loading the dataset
train_LR = pd.read_parquet("train_feature_engineered.parquet")
val_LR = pd.read_parquet("val_feature_engineered.parquet")

In [11]:

#All the price features which include values as medium and medium_low
valuation_features = [
    'Shillers Cyclically Adjusted P/E Ratio',
    'Enterprise Value Multiple',
    'Price/Book',
    'Price/Sales',
    'Price/Cash flow',
    'Price/Operating Earnings (Basic, Excl. EI)',
    'Price/Operating Earnings (Diluted, Excl. EI)',
    'P/E (Diluted, Excl. EI)',
    'P/E (Diluted, Incl. EI)',
    'Trailing P/E to Growth (PEG) ratio',
    'Book/Market',
    'Dividend Yield',
    'Dividend Payout Ratio'
]

# Select _Regime columns
regime_cols = [f"{col}_Regime" for col in valuation_features if f"{col}_Regime" in train_LR.columns]

# Convert to string
for col in regime_cols:
    train_LR[col] = train_LR[col].astype(str)
    val_LR[col] = val_LR[col].astype(str)

# Missing values
for col in regime_cols:
    train_LR[col] = train_LR[col].replace("nan", "Missing")
    val_LR[col] = val_LR[col].replace("nan", "Missing")

# One-hot encoding (
train_LR = pd.get_dummies(train_LR, columns=regime_cols, drop_first=True)
val_LR = pd.get_dummies(val_LR, columns=regime_cols, drop_first=True)

# Align val and test on train columns
val_LR = val_LR.reindex(columns=train_LR.columns, fill_value=0)


In [12]:
#making 3 class return
# Train
conditions_train_LR = [
    (train_LR['Return'] > 3),
    (train_LR['Return'] >= -3) & (train_LR['Return'] <= 3),
    (train_LR['Return'] < -3)
]

choices = [1, 0, -1]

train_LR['Return_class_3'] = np.select(conditions_train_LR, choices, default=np.nan)


# Validation
conditions_val_LR = [
    (val_LR['Return'] > 3),
    (val_LR['Return'] >= -3) & (val_LR['Return'] <= 3),
    (val_LR['Return'] < -3)
]

val_LR['Return_class_3'] = np.select(conditions_val, choices, default=np.nan)

print("--- Train distribution (3-class) ---")
print(train_LR['Return_class_3'].value_counts())
print(train_LR['Return_class_3'].value_counts(normalize=True))

print("\n--- Validation distribution (3-class) ---")
print(val_LR['Return_class_3'].value_counts())
print(val_LR['Return_class_3'].value_counts(normalize=True))


--- Train distribution (3-class) ---
Return_class_3
 0.0    5749
 1.0    1728
-1.0    1551
Name: count, dtype: int64
Return_class_3
 0.0    0.636797
 1.0    0.191405
-1.0    0.171799
Name: proportion, dtype: float64

--- Validation distribution (3-class) ---
Return_class_3
 0.0    1617
 1.0     329
-1.0     297
Name: count, dtype: int64
Return_class_3
 0.0    0.720909
 1.0    0.146679
-1.0    0.132412
Name: proportion, dtype: float64


In [13]:
#making a return class
target = "Return"

X_train_LR = train_LR.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_train_LR = train_LR["Return_class_3"]

X_val_LR = val_LR.drop(columns=["Return","Return_class","EPS_Estimate","EPS_Actual","Close_Before","Open_After","Return_class_3"])
y_val_LR = val_LR["Return_class_3"]

In [14]:
#dropping columns
drop_cols = [
    'CUSIP',
    'Global Company Key',
    'Historical CRSP PERMNO Link to COMPUSTAT Record',
    'Ticker',
    'Date',
    'EarningsDate',
    'Fiscal year end',
    'Fiscal quarter end',
    'Year',
    'Month'
]

X_train_LR = X_train_LR.drop(columns=[c for c in drop_cols if c in X_train_LR.columns])
X_val_LR = X_val_LR.drop(columns=[c for c in drop_cols if c in X_val_LR.columns])

In [44]:
#Making imputer
imputer = SimpleImputer(strategy='mean')

# Fit on training
X_train_scaled = imputer.fit_transform(X_train_LR)

# Transform val
X_val_scaled = imputer.transform(X_val_LR)


In [46]:
X_all_LR = pd.concat([X_train_LR, X_val_LR], axis=0)
y_all_LR = pd.concat([y_train_LR, y_val_LR], axis=0).reset_index(drop=True)

In [49]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_pred_lr_oof = []
y_true_lr_oof = []

imputer = SimpleImputer(strategy="median")

X_all_LR = pd.DataFrame(
    imputer.fit_transform(X_all_LR),
    columns=X_all_LR.columns
)

for train_index, val_index in skf.split(X_all, y_all):

    X_tr = X_all_LR.iloc[train_index]
    X_val_fold = X_all_LR.iloc[val_index]

    y_tr = y_all_LR.iloc[train_index]
    y_val_fold = y_all_LR.iloc[val_index]

    # scaling ONLY inside fold (correct!)
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_val_fold = scaler.transform(X_val_fold)

    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr, y_tr)

    y_pred = model.predict(X_val_fold)

    y_pred_lr_oof.extend(y_pred)
    y_true_lr_oof.extend(y_val_fold.values)

In [50]:
print(len(y_true_lr_oof))
print(len(y_true_rf_oof))
print(len(y_true_xgb_oof))

11271
11271
11271


In [51]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(y_true, y_pred_1, y_pred_2, name1, name2):

    y_true = np.array(y_true)
    y_pred_1 = np.array(y_pred_1)
    y_pred_2 = np.array(y_pred_2)

    m1_correct = (y_pred_1 == y_true)
    m2_correct = (y_pred_2 == y_true)

    table = np.array([
        [np.sum(m1_correct & m2_correct), np.sum(m1_correct & ~m2_correct)],
        [np.sum(~m1_correct & m2_correct), np.sum(~m1_correct & ~m2_correct)]
    ])

    result = mcnemar(table, exact=True)

    print(f"\n=== {name1} vs {name2} ===")
    print(table)
    print("p-value:", result.pvalue)

    if result.pvalue < 0.05:
        print("➡ Significant verschil")
    else:
        print("➡ Geen significant verschil")

    return result

In [52]:
mcnemar_test(y_true_lr_oof, y_pred_lr_oof, y_pred_rf_oof,
             "Logistic Regression", "Random Forest")

mcnemar_test(y_true_lr_oof, y_pred_lr_oof, y_pred_xgb_oof,
             "Logistic Regression", "XGBoost")


=== Logistic Regression vs Random Forest ===
[[1592 5873]
 [ 654 3152]]
p-value: 0.0
➡ Significant verschil

=== Logistic Regression vs XGBoost ===
[[ 263 7202]
 [1320 2486]]
p-value: 0.0
➡ Significant verschil


<bunch containing results, print to see contents>